In [1]:
# Install dependencies (v2)
!pip install FlagEmbedding faiss-cpu numpy pandas -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 124.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.6 MB/s eta 0:00:00


In [2]:
# FlagEmbedding + faiss-cpu installed in Block 1 above
# faiss-cpu: exact cosine sim, same results as numpy
# Switch to faiss at >1M candidates or repeated queries


In [3]:
# faiss-cpu optional — see note in rank.py
# For 100K candidates, numpy cosine sim is fast enough (<1s)
# Switch to faiss when: dataset > 1M, or repeated queries needed

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

# ── v2 paths ──────────────────────────────────────────────────────────────
DRIVE_DIR       = '/content/drive/MyDrive/redrob'
CANDIDATES_PATH = f'{DRIVE_DIR}/candidates.jsonl'
SAMPLE_PATH     = f'{DRIVE_DIR}/sample_candidates.json'
ARTIFACTS_DIR   = f'{DRIVE_DIR}/artifacts_v2'   # v2: separate from v1 artifacts

# Verify input files exist
for path in [CANDIDATES_PATH, SAMPLE_PATH]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1e6 if exists else 0
    print(f"{'✓' if exists else '✗'} {path} ({size:.1f} MB)")


✓ /content/drive/MyDrive/redrob/candidates.jsonl (487.3 MB)
✓ /content/drive/MyDrive/redrob/sample_candidates.json (0.3 MB)


In [6]:
import json
import numpy as np
from collections import Counter
from datetime import datetime

TODAY = datetime(2026, 6, 1)

# Load sample for analysis — fast way to understand data before full 100K run
with open(SAMPLE_PATH) as f:
    sample = json.load(f)

print(f"Sample size: {len(sample)} candidates")
print(f"Top-level keys: {list(sample[0].keys())}")
print(f"Profile keys: {list(sample[0]['profile'].keys())}")
print(f"redrob_signals keys: {list(sample[0]['redrob_signals'].keys())}")

# ── HARD FILTER SIGNALS ──────────────────────────────────────────────────────
# These are binary flags — we reject if false (except linkedin which is a bonus)
print("\n=== HARD FILTER SIGNALS ===")
open_to_work   = sum(1 for c in sample if c['redrob_signals']['open_to_work_flag'])
verified_email = sum(1 for c in sample if c['redrob_signals']['verified_email'])
verified_phone = sum(1 for c in sample if c['redrob_signals']['verified_phone'])
linkedin       = sum(1 for c in sample if c['redrob_signals']['linkedin_connected'])

print(f"open_to_work_flag = True : {open_to_work}/{len(sample)}  → reject if False")
print(f"verified_email    = True : {verified_email}/{len(sample)}  → reject only if BOTH false")
print(f"verified_phone    = True : {verified_phone}/{len(sample)}  → reject only if BOTH false")
print(f"linkedin_connected= True : {linkedin}/{len(sample)}  → bonus in behavioral score")

# ── NOTICE PERIOD ─────────────────────────────────────────────────────────────
# >60 days = hard reject. 31-60 = penalty. <=30 = preferred
print("\n=== NOTICE PERIOD ===")
notice = [c['redrob_signals']['notice_period_days'] for c in sample]
print(f"min:{min(notice)}  max:{max(notice)}  median:{sorted(notice)[len(notice)//2]}")
print(f"  <=30d (preferred)  : {sum(1 for n in notice if n<=30)}")
print(f"  31-60d (penalty)   : {sum(1 for n in notice if 30<n<=60)}")
print(f"  >60d (hard reject) : {sum(1 for n in notice if n>60)}")

# ── INDUSTRY & TITLE ──────────────────────────────────────────────────────────
# Manufacturing/Paper Products = hard reject
# Non-tech titles = hard reject regardless of industry
print("\n=== INDUSTRY DISTRIBUTION ===")
industries = Counter(c['profile']['current_industry'] for c in sample)
for ind, cnt in industries.most_common(10):
    print(f"  {ind}: {cnt}")

print("\n=== TITLE DISTRIBUTION ===")
titles = Counter(c['profile']['current_title'] for c in sample)
for title, cnt in titles.most_common(10):
    print(f"  {title}: {cnt}")

# ── PROFILE COMPLETENESS ──────────────────────────────────────────────────────
# Decision: NOT a hard filter — used as scoring signal directly
# Reason: 5,761 candidates would be rejected at <50% threshold — too aggressive
# Instead: redrob's own score feeds into feature scoring (0.1 to 1.0)
print("\n=== PROFILE COMPLETENESS ===")
completeness = [c['redrob_signals']['profile_completeness_score'] for c in sample]
print(f"min:{min(completeness):.1f}  max:{max(completeness):.1f}  median:{sorted(completeness)[len(completeness)//2]:.1f}")
print(f"  Would reject if <50% threshold: {sum(1 for s in completeness if s<50)} in this sample")
print("  → Used as scoring signal instead (see feature_score in rank.py)")

# ── SALARY ───────────────────────────────────────────────────────────────────
# Full dataset max is 74 LPA — >60 LPA gets mild penalty (x0.90)
# No hard reject on salary — just misalignment signal
print("\n=== SALARY EXPECTATION (max LPA) ===")
salaries = [c['redrob_signals']['expected_salary_range_inr_lpa']['max'] for c in sample]
print(f"min:{min(salaries):.0f}  max:{max(salaries):.0f}")
print(f"  >60 LPA (mild penalty x0.90): {sum(1 for s in salaries if s>60)}")
print("  NOTE: full 100K dataset max is 74 LPA — nothing above 80 LPA exists")

# ── HONEYPOT DETECTION ────────────────────────────────────────────────────────
# Per spec: "good ranking system naturally avoids them"
# Two profile integrity checks catch them without special-casing

# Check 1: any skill with duration_months = 0
# Impossible to claim proficiency with zero usage
zero_dur = sum(
    1 for c in sample
    if any(s.get('duration_months', 1) == 0 for s in c.get('skills', []))
)
print(f"\n=== HONEYPOT SIGNALS ===")
print(f"Skill duration=0 (integrity fail) : {zero_dur}/{len(sample)}")

# Check 2: experience sum mismatch > 6 months
# In full 100K: mismatches jump from ~6mo to 130+mo — natural cliff
# No candidates exist between 6 and 24 months mismatch
mismatches = []
for c in sample:
    total   = sum(j.get('duration_months', 0) for j in c.get('career_history', []))
    claimed = c['profile']['years_of_experience'] * 12
    mismatches.append(abs(total - claimed))
print(f"Exp sum mismatch >6mo           : {sum(1 for m in mismatches if m>6)}/{len(sample)}")
print("  NOTE: full 100K shows cliff — 49 candidates with 130-185mo mismatch, none between 6-24mo")

# ── SKILL ASSESSMENTS ─────────────────────────────────────────────────────────
# redrob has assessed candidates on specific skills (0-100 score)
# We use JD-relevant assessments as bonus in feature scoring
# CV/speech assessments (YOLO, CNN etc.) are penalty signals
print("\n=== SKILL ASSESSMENT CATEGORIES (sample) ===")
all_assessments = Counter()
for c in sample:
    for k in c['redrob_signals'].get('skill_assessment_scores', {}).keys():
        all_assessments[k] += 1
print(f"Unique assessment categories: {len(all_assessments)}")
for k, v in all_assessments.most_common(15):
    print(f"  {k}: {v}")

Sample size: 50 candidates
Top-level keys: ['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals']
Profile keys: ['anonymized_name', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title', 'current_company', 'current_company_size', 'current_industry']
redrob_signals keys: ['profile_completeness_score', 'signup_date', 'last_active_date', 'open_to_work_flag', 'profile_views_received_30d', 'applications_submitted_30d', 'recruiter_response_rate', 'avg_response_time_hours', 'skill_assessment_scores', 'connection_count', 'endorsements_received', 'notice_period_days', 'expected_salary_range_inr_lpa', 'preferred_work_mode', 'willing_to_relocate', 'github_activity_score', 'search_appearance_30d', 'saved_by_recruiters_30d', 'interview_completion_rate', 'offer_acceptance_rate', 'verified_email', 'verified_phone', 'linkedin_connected']

=== HARD FILTER SIGNALS ===
open_to_work_flag = True : 16/50  → reject 

In [7]:
from datetime import datetime

# All outputs saved to Drive — survives Colab session reset
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M')

# v2 folder structure
ARTIFACTS_DIR = f'{DRIVE_DIR}/artifacts_v2'          # embeddings + metadata
OUTPUT_DIR    = f'{DRIVE_DIR}/outputs_v2/{RUN_ID}'   # submission CSV per run
LOG_DIR       = f'{OUTPUT_DIR}/logs'

for d in [ARTIFACTS_DIR, OUTPUT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'✓ {d}')

print(f'\nRUN_ID: {RUN_ID}')

# Save config to Drive — survives reconnect
config = f'''# v2 config — {RUN_ID}
RUN_ID        = '{RUN_ID}'
DRIVE_DIR     = '{DRIVE_DIR}'
ARTIFACTS_DIR = '{DRIVE_DIR}/artifacts_v2'
OUTPUT_DIR    = '{DRIVE_DIR}/outputs_v2/{RUN_ID}'
CANDIDATES_PATH = '{DRIVE_DIR}/candidates.jsonl'
SAMPLE_PATH     = '{DRIVE_DIR}/sample_candidates.json'
'''
with open(f'{DRIVE_DIR}/config_v2.py', 'w') as f:
    f.write(config)
print(f'✓ Config saved to {DRIVE_DIR}/config_v2.py')
print(f'  If Colab disconnects: exec(open("{DRIVE_DIR}/config_v2.py").read())')


✓ /content/drive/MyDrive/redrob/artifacts_v2
✓ /content/drive/MyDrive/redrob/outputs_v2/20260627_0506
✓ /content/drive/MyDrive/redrob/outputs_v2/20260627_0506/logs

RUN_ID: 20260627_0506
✓ Config saved to /content/drive/MyDrive/redrob/config_v2.py
  If Colab disconnects: exec(open("/content/drive/MyDrive/redrob/config_v2.py").read())


In [8]:
import zipfile

# Unzip codes_v2.zip — contains precompute_v2.py and rank_v2.py
zip_path = f'{DRIVE_DIR}/codes_v2.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(DRIVE_DIR)
    print(f'Extracted: {z.namelist()}')

# Verify v2 files
for f in ['precompute_v2.py', 'rank_v2.py']:
    path = f'{DRIVE_DIR}/{f}'
    size = os.path.getsize(path)/1e3 if os.path.exists(path) else 0
    print(f"{'✓' if os.path.exists(path) else '✗'} {f} ({size:.1f} KB)")


Extracted: ['precompute_v2.py', 'rank_v2.py']
✓ precompute_v2.py (26.8 KB)
✓ rank_v2.py (37.1 KB)


In [9]:
# Verify v2 files are present and sized correctly
for f in ['precompute_v2.py', 'rank_v2.py']:
    path = f'{DRIVE_DIR}/{f}'
    size = os.path.getsize(path)/1e3 if os.path.exists(path) else 0
    print(f"{'✓' if os.path.exists(path) else '✗'} {f} ({size:.1f} KB)")


✓ precompute_v2.py (26.8 KB)
✓ rank_v2.py (37.1 KB)


In [10]:
import subprocess, sys

# Run precompute_v2 on SAMPLE first — 50 candidates, fast
# Verifies all v2 changes work before touching full 100K
result = subprocess.run([
    sys.executable, f'{DRIVE_DIR}/precompute_v2.py',
    '--input',      SAMPLE_PATH,
    '--out_dir',    ARTIFACTS_DIR,
    '--batch_size', '16'
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('ERROR:')
    print(result.stderr)



Redrob Precompute Pipeline v2
Input     : /content/drive/MyDrive/redrob/sample_candidates.json
Output    : /content/drive/MyDrive/redrob/artifacts_v2
Batch size: 16

Loading candidates...
Loaded 50 candidates

Extracting metadata...
Metadata extracted for 50 candidates

Building candidate texts (full summary + career descriptions)...
Average text length: 2171 chars (~543 tokens)

Loading BGE-M3 model...
Model loaded

Generating JD embedding (P7: real JD, not centroid proxy)...
  Using built-in JD text (1010 chars)
  jd_embedding.npy saved — shape: (1024,)

Generating candidate embeddings (batch_size=16)...
  32.0% — 16/50 — 0s elapsed

Embedding complete — shape: (50, 1024) — 0s total

Saving artifacts...
  embeddings.npy  — 0.2 MB
  metadata.pkl    — 0.1 MB
  jd_embedding.npy — 4.2 KB

Precompute v2 complete!
  Candidates processed : 50
  Artifacts saved to   : /content/drive/MyDrive/redrob/artifacts_v2
  New in v2: full summary, full career text, real JD embedding,
             avg_

In [11]:
import numpy as np, pickle

emb_path  = f'{ARTIFACTS_DIR}/embeddings.npy'
meta_path = f'{ARTIFACTS_DIR}/metadata.pkl'
jd_path   = f'{ARTIFACTS_DIR}/jd_embedding.npy'

emb  = np.load(emb_path)
jd   = np.load(jd_path)
with open(meta_path, 'rb') as f:
    meta = pickle.load(f)

print(f'✓ embeddings.npy   — shape: {emb.shape} — {os.path.getsize(emb_path)/1e6:.1f} MB')
print(f'✓ metadata.pkl     — {len(meta)} candidates — {os.path.getsize(meta_path)/1e6:.1f} MB')
print(f'✓ jd_embedding.npy — shape: {jd.shape} — norm: {np.linalg.norm(jd):.4f}')

# Verify v2 fields in metadata
m = meta[0]
print(f'\nv2 metadata fields check:')
print(f'  full_summary len  : {len(m.get("full_summary", ""))} chars (was 200 in v1)')
print(f'  avg_tenure_months : {m.get("avg_tenure_months", "MISSING")}')
print(f'  summary_quality   : {m.get("summary_quality", "MISSING")}')
print(f'  num_companies     : {m.get("num_companies", "MISSING")}')
print(f'  company_size_score: {m.get("company_size_score", "MISSING")}')
print(f'  is_faang_only     : {m.get("is_faang_only", "MISSING")}')
print(f'  location_bucket   : {m.get("location_bucket", "MISSING")}')
print(f'  candidate_id      : {m["candidate_id"]}')


✓ embeddings.npy   — shape: (50, 1024) — 0.2 MB
✓ metadata.pkl     — 50 candidates — 0.1 MB
✓ jd_embedding.npy — shape: (1024,) — norm: 1.0000

v2 metadata fields check:
  full_summary len  : 634 chars (was 200 in v1)
  avg_tenure_months : 41.0
  summary_quality   : 0.7333333333333333
  num_companies     : 2
  company_size_score: 0.5
  is_faang_only     : False
  location_bucket   : outside_india
  candidate_id      : CAND_0000001


In [12]:
# Full 100K run on A100 GPU
# precompute_v2 changes: full summary, full career text,
# real JD embedding, avg_tenure, company_size, FAANG flag
# Expected time: ~30 min on A100, ~2.5hr on T4
print('Starting full precompute_v2 on 100K candidates...')
print('Do NOT close Colab or let it idle — keep the tab active\n')

result = subprocess.run([
    sys.executable, f'{DRIVE_DIR}/precompute_v2.py',
    '--input',      CANDIDATES_PATH,
    '--out_dir',    ARTIFACTS_DIR,   # saves to artifacts_v2/
    '--batch_size', '512'             #
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('ERROR:')
    print(result.stderr)


Starting full precompute_v2 on 100K candidates...
Do NOT close Colab or let it idle — keep the tab active


Redrob Precompute Pipeline v2
Input     : /content/drive/MyDrive/redrob/candidates.jsonl
Output    : /content/drive/MyDrive/redrob/artifacts_v2
Batch size: 512

Loading candidates...
Loaded 100,000 candidates

Extracting metadata...
  10,000 / 100,000
  20,000 / 100,000
  30,000 / 100,000
  40,000 / 100,000
  50,000 / 100,000
  60,000 / 100,000
  70,000 / 100,000
  80,000 / 100,000
  90,000 / 100,000
Metadata extracted for 100,000 candidates

Building candidate texts (full summary + career descriptions)...
Average text length: 2193 chars (~548 tokens)

Loading BGE-M3 model...
Model loaded

Generating JD embedding (P7: real JD, not centroid proxy)...
  Using built-in JD text (1010 chars)
  jd_embedding.npy saved — shape: (1024,)

Generating candidate embeddings (batch_size=512)...
  0.5% — 512/100,000 — 6s elapsed
  5.6% — 5,632/100,000 — 70s elapsed
  10.8% — 10,752/100,000 — 131